# Ch 8.4 Sequences

A sequence is a **delayed list**. A sequence could be finite, like a list, but could also be infinite.

Sequences can be persistent or ephemeral. https://v2.ocaml.org/api/Seq.html

Key concept: **Lazy evaluation**!

**NOTE:** Output on this Jupyter Notebook page has been cleared intentionally because many cells will produce extremely long outputs that reach the maximum that a call can hold.

In [10]:
(* Recursivie values *)
let rec ones = 1 :: ones;;
let rec a = 0 :: b and b = 1 :: a;; (* a : [0; 1; 0; 1; 0; 1; 0; 1; ...] *)

val ones : int list = [1; <cycle>]


val a : int list = [0; 1; <cycle>]
val b : int list = [1; 0; <cycle>]


#### The following non-converging recursive definition DOES NOT work for sequences

In [11]:
(* How NOT to define a sequence of ones *)

(** [from n] is the infinite list [[n; n + 1; n + 2; ...]]. *)
let rec ones_fun n = 1 :: ones_fun (n + 1)

(** [nats] is the infinite list of natural numbers [[0; 1; ...]]. *)
let rec from n = n :: from (n + 1)

val ones_fun : int -> int list = <fun>


val from : int -> int list = <fun>


In [ ]:
let ones = ones_fun 0

In [ ]:
(** [nats] is the infinite list of natural numbers [[0; 1; ...]]. *)
let nats = from 0

In [ ]:
(* Another clever way of defining natural numbers that won't work. *)
let rec nats = 0 :: List.map (fun x -> x + 1) nats

#### The following list definition method DOES NOT work for sequences

In [ ]:
(* This method works for finite lists but not infinite sequences. *)

(* Finite lists *)
type 'a mylist = Nil | Cons of 'a * 'a mylist

(* Infinite sequences *)
type 'a sequence = Cons of 'a * 'a sequence

In [ ]:
let rec from n = Cons (n, from (n + 1))
let nats = from 0 (* This still doesn't work *)

### Lazy Evaluation

In [ ]:
let f1 = failwith "oops"

In [ ]:
let f2 = fun x -> failwith "oops"

In [ ]:
f2 ();;

## Sequences with lazy evaluation. This works!

In [ ]:
(* This works. Lazy evaluation. *)
(** An ['a sequence] is an infinite list of values of type ['a].
    AF (Abstract Function): [Cons (x, f)] is the sequence whose head is [x] and tail is [f ()].
    RI (Recursive Invariant): none. *)
type 'a sequence = Cons of 'a * (unit -> 'a sequence)
let rec from n = Cons (n, fun () -> from (n + 1))
let nats = from 0

In [ ]:
let nats1 = from 1;;
let nats100 = from 100;;

match nats100 with
| Cons (x1, f1) -> x1;;

#### QUESTION: How many function calls are needed to compute 100 from nats100?
#### QUESTION: How many function calls are needed to compute 101 from nats100?
#### QUESTION: How many function calls are needed to compute 102 from nats100?

Hint: count the number of () in the code.

In [ ]:
match nats100 with
| Cons (x1, f1) -> match f1() with
                  | Cons (x2, f2) -> x2;;

 match nats100 with
               | Cons (x1, f1) -> match f1() with
                                 | Cons (x2, f2) -> match f2() with
                                          | Cons (x3, f3) -> x3;;
                  

#### Luckily, everything for sequences has been implemented for you.

In [ ]:
open Seq

let sr = forever Random.bool (* sr: sequence of random booleans *)


In [ ]:
let id x = x

In [ ]:
let s2 = init 10 id

(* `init n f` is the finite sequence f 0; f 1; ...; f (n-1). *)
(* https://v2.ocaml.org/api/Seq.html *)
(* If desired, the infinite sequence f 0; f 1; ... can be defined as map f (ints 0). *)

In [ ]:
let s_infinite = map id (ints 0)

(* `ints i` is the infinite sequence of the integers beginning at i and counting up. *)
(* https://v2.ocaml.org/api/Seq.html *)

(* As a general rule, the functions that build sequences, such as map, filter, scan, take, etc., 
   produce sequences whose elements are computed only on demand. 
   The functions that eagerly consume sequences, such as is_empty, find, length, iter, fold_left, etc., 
   are the functions that force computation to take place.  *)

In [ ]:
length s2

In [ ]:
let p x = print_int x; print_endline ""; flush_all();;

In [ ]:
iter p s2

In [ ]:
let s3 = init 100 id

In [ ]:
iter p s3

In [ ]:
let s5 = init 100000 id;;
length s5

In [ ]:
let s6 = init 1000000 id;;
length s6;;
let s7 = init 10000000 id;;
length s7

In [ ]:
let s8 = init 50000000 id;;
length s8;;

(* Re-running this will not shrink the execution time from 2s to 0.0s. *)

In [ ]:
length s8;;

In [ ]:
(* let rec size_list lst = 
  match lst with
  | [] -> 0
  | h :: t -> (size_list t) + 1
*)

let rec size s = 
  match s with
  | Nil -> 0
  | Cons (x, xs) -> (size (xs ())) + 1

In [ ]:
size (s3 ())

In [ ]:
size (s5 ())

In [ ]:
size (s6 ())

In [ ]:
size (s7 ())

In [ ]:
open Seq
let r () = Random.int  100
let si = forever r

(* `forever f` is an infinite sequence where every element is produced (on demand) by the function call f(). *)
(* Seq.forever https://v2.ocaml.org/api/Seq.html *)

In [ ]:
length si (* This takes forever, literally. *)

In [ ]:
iter p si (* This will also take forever. *)

In [ ]:
open Seq

In [ ]:
module MySeq = Seq


### More examples of operations on sequences

https://cs3110.github.io/textbook/chapters/ds/sequence.html

In [1]:
(** An ['a sequence] is an infinite list of values of type ['a].
    AF: [Cons (x, f)] is the sequence whose head is [x] and tail is [f ()].
    RI: none. *)
type 'a sequence = Cons of 'a * (unit -> 'a sequence)

type 'a sequence = Cons of 'a * (unit -> 'a sequence)


In [2]:
let rec from n = Cons (n, fun () -> from (n + 1))
let nats = from 0

val from : int -> int sequence = <fun>


val nats : int sequence = Cons (0, <fun>)


In [3]:
(** [hd s] is the head of [s] *)
let hd (Cons (h, _)) = h

(*
let hd x = 
    match x with
    | [] -> failwith "empty sequence"
    | Cons( h, _) -> h
*)

val hd : 'a sequence -> 'a = <fun>


In [4]:
(** [tl s] is the tail of [s] *)
let tl (Cons (_, f_t)) = f_t ()

val tl : 'a sequence -> 'a sequence = <fun>


In [5]:
(** [take n s] is the list of the first [n] elements of [s] *)
let rec take n s =
  if n = 0 then [] else hd s :: take (n - 1) (tl s)

(** [drop n s] is all but the first [n] elements of [s] *)
let rec drop n s =
  if n = 0 then s else drop (n - 1) (tl s)

val take : int -> 'a sequence -> 'a list = <fun>


val drop : int -> 'a sequence -> 'a sequence = <fun>


In [6]:
take 10 nats

- : int list = [0; 1; 2; 3; 4; 5; 6; 7; 8; 9]


In [7]:
drop 10 nats

- : int sequence = Cons (10, <fun>)


In [8]:
drop 10 (drop 10 nats)

- : int sequence = Cons (20, <fun>)


In [9]:
take 10 (drop 10 (drop 10 nats))

- : int list = [20; 21; 22; 23; 24; 25; 26; 27; 28; 29]


In [ ]:
take 1000000 nats

### Exercise: How to turn "take" into a tail recursive one?

### Exercise: Implement a function 'element n' that returns the nth element of a sequence?

using `drop`, this is trivial. How to implement it without drop?    

In [ ]:
let rec element n s =
  match drop n s with 
  | Cons (h, t) -> h

In [ ]:
element 3 nats;;
element 1000000 nats;;

